# Los cinco niveles de agencia

**Sesión 1 · Agentes y arquitecturas multiagénticas**
Programa de formación en IA · Qypher para Grupo Bios

---

> ### ⚠ Datos sintéticos
>
> Todos los datos de `bios_ops.db` son **sintéticos**, generados por
> `backend/db/seed.py`. Los nombres de plantas, productos, equipos y clientes son
> **ficticios** y no representan la red de operaciones, los proveedores ni la
> cartera de clientes de Grupo Bios. Los municipios son reales; la asignación
> planta↔municipio es inventada.
>
> **Ningún dato real de la compañía se procesa en esta sesión.** La razón es
> operativa, no ceremonial: este laboratorio envía el contenido de las
> herramientas a un proveedor externo de LLM. Hacerlo con datos productivos exige
> antes definir contrato de tratamiento, clasificación de la información y
> controles de retención — trabajo de las sesiones 2 y 7.

---

## Cómo funciona este notebook

Construyes los cinco niveles de agencia del ebook, de menor a mayor. Cada nivel
tiene **cinco celdas**: te explico, construyes, corres y verificas, observamos, y
un reto opcional.

**Tres reglas:**

1. **Cada sección se puede hacer sin haber terminado la anterior.** Al inicio de
   cada una hay una celda de rescate plegada con la implementación de referencia
   del nivel previo. Si te trabas, la ejecutas y sigues. No pierdas la sesión por
   un nivel.
2. **Lo único que escribes son los `# TODO`.** Los imports, el cliente del
   modelo, la impresión de trazas y los `assert` vienen hechos. En 55 minutos no
   hay tiempo para plomería, y la plomería no es el tema.
3. **Los `assert` te dicen si terminaste.** No necesitas preguntar «¿así está
   bien?»: la celda de verificación te responde.

| Nivel | Patrón | Líneas que escribes |
|---|---|---|
| N1 · Procesador simple ☆☆☆ | `process_llm_output(llm_response)` | 0 — solo observas |
| N2 · Enrutador ★☆☆ | `if llm_decision(): path_a() else: path_b()` | 2 |
| N3 · Llamador de herramientas ★★☆ | `run_function(llm_chosen_tool, llm_chosen_args)` | 4 |
| N4 · Agente multipasos ★★★ | `while llm_should_continue(): execute_next_step()` | 3 |
| N5 · Sistema multiagente ★★★★ | `if llm_trigger(): execute_agent()` | 4 |

---

## 0 · Preparación

Ejecuta las dos celdas siguientes. Deben tardar menos de diez segundos.

In [ ]:
# Todo lo que necesitas, de una vez. No hace falta volver a esta celda.
import json
import warnings
from typing import Literal

from pydantic import BaseModel, Field

# En LangGraph 1.x, `create_react_agent` está DEPRECADA en favor de
# `langchain.agents.create_agent` (se elimina en la 2.0). Usamos la del ebook a
# propósito: es el nombre que tienes en el material de clase. Silenciamos el aviso
# para que no aparezca en cada celda — pero queda dicho acá, no escondido.
warnings.filterwarnings("ignore", message=".*create_react_agent.*")
from langgraph.prebuilt import create_react_agent

from backend.config import config
from backend.db import consulta_directa
from backend.eventos import mensajes_publicables
from backend.lab import (
    PREGUNTA,            # la pregunta insignia, la misma para los cinco niveles
    PREGUNTA_CRUZADA,    # la de N5: cruza dos dominios
    correr,              # ejecuta un nivel y devuelve su traza
    ultima_traza,        # la traza de lo último que corriste
    valor_real_inventario,
)
from backend.llm import (
    cliente,             # cliente de chat (LangChain)
    cliente_crudo,       # cliente de OpenAI sin envolturas — para ver el JSON crudo
    con_reintentos,      # backoff con jitter: la key es compartida
    esquemas_openai,     # funciones Python -> esquema de tools de OpenAI
    estado,
    verificar_entorno,
)
from backend.niveles.base import NivelBase
from backend.niveles.multiagente import Delegador, SubAgente
from backend.niveles.react import stream_react
from backend.tools.operaciones import (
    POR_NOMBRE,          # nombre de tool -> función Python
    TODAS as TOOLS,      # las 7 tools de dominio
    TOOLS_ABASTECIMIENTO,
    TOOLS_OPERACIONES,
)

estado()
print(f"\n{len(TOOLS)} tools disponibles: {', '.join(POR_NOMBRE)}")
print(f"\nPregunta insignia:\n  «{PREGUNTA}»")

In [ ]:
# Si algo falla acá, el mensaje dice qué hacer. Esto es lo mismo que corriste en
# el preflight de la semana pasada.
verificar_entorno()

### El dato de verdad

Antes de empezar, mira el dato real, consultado directamente contra la base sin
pasar por ningún modelo. **Guárdalo**: en un minuto vas a comparar la respuesta de
un LLM contra esto.

In [ ]:
valor_real_inventario()

---

## 1 · N1 · Procesador simple ☆☆☆

> ⏱ **Minuto 0 de 55.**

**Patrón:** `process_llm_output(llm_response)`

Una llamada al modelo. Sin herramientas. Sin decisiones. El nivel cero de la
agencia: el modelo recibe texto y devuelve texto.

El system prompt lo presenta como asistente de operaciones de una planta. No se le
pide inventar nada — se le da un rol y se observa qué hace **sin acceso a ninguna
fuente**.

Aquí no escribes nada. Ejecuta y observa.

In [ ]:
class MiN1(NivelBase):
    ID = "n1"
    NOMBRE = "Procesador simple"
    ESTRELLAS = "☆☆☆"
    PATRON = "process_llm_output(llm_response)"

    async def correr(self, ej, pregunta):
        mensajes = [
            {"role": "system", "content": self.system_prompt()},
            {"role": "user", "content": pregunta},
        ]

        yield ej.ev(
            "llm_request",
            n_llamada=ej.proxima_llamada(),
            mensajes=mensajes_publicables(mensajes),
            tools_declaradas=[],          # ← vacío. Ese es el nivel.
        )

        respuesta = await con_reintentos(lambda: cliente().ainvoke(mensajes))

        uso = respuesta.usage_metadata or {}
        ej.contar_tokens(uso.get("input_tokens", 0), uso.get("output_tokens", 0))
        texto = respuesta.content

        yield ej.ev(
            "llm_response",
            n_llamada=1,
            texto=texto,
            hay_tool_calls=False,
            tokens_in=uso.get("input_tokens", 0),
            tokens_out=uso.get("output_tokens", 0),
            ms=0,
        )

        ej.respuesta_final = texto
        yield ej.ev("respuesta_final", texto=texto)


traza = await correr(MiN1(), PREGUNTA)

In [ ]:
# --- Corro y verifico ---
traza = ultima_traza()

assert traza.llamadas_llm == 1, f"N1 hace exactamente 1 llamada, hizo {traza.llamadas_llm}"
assert traza.llamadas_tools == 0, "N1 no tiene herramientas: no puede llamar ninguna"

real = valor_real_inventario()
print(f"El modelo dijo : «{traza.respuesta_final}»")
print(f"\nLa base dice   : {real['cantidad_ton']} t de {real['materia_prima']} "
      f"en {real['planta']} (mínimo {real['stock_minimo_ton']} t)")

if str(int(real["cantidad_ton"])) in traza.respuesta_final:
    print("\n… acertó por casualidad. Vuelve a correr la celda anterior.")
else:
    print("\n→ Lo que dijo el modelo no viene de ninguna fuente.")

print("\n✓ N1 correcto")

### Observo

1. Si el modelo dio una cifra: **¿de dónde salió?** No de la base — no la puede
   consultar. ¿Cambiaría algo si el prompt dijera «no inventes»?
2. Si el modelo se negó a responder: se portó bien, pero **¿le sirve a alguien de
   la operación?**

*(En el tablero proyectado, este es el momento del aviso ⚠ «afirmó una cantidad
sin haber consultado ninguna fuente». Es una heurística por expresión regular, no
un detector de alucinaciones — y por eso hay además un botón manual.)*

In [ ]:
# --- Reto (opcional, para después de la sesión) ---
#
# Escribe un system prompt que le PROHÍBA inventar cifras y mide cuántas veces de
# 10 falla igual.
#
#   from backend import prompts
#   prompts.fijar("n1", "…tu prompt…")
#   for i in range(10):
#       t = await correr(MiN1(), PREGUNTA, imprimir=False)
#       print(i, t.respuesta_final[:80])
#   prompts.restaurar()
#
# [NÚCLEO] ¿Cuál es la tasa de fallo con tu mejor prompt? ¿Baja de cero alguna vez?
# Muéstralo a tu mesa.

---

## 2 · N2 · Enrutador ★☆☆

> ⏱ **Minuto 4 de 55.** Si tu N1 no corrió, no importa: N2 no depende de él.

**Patrón:** `if llm_decision(): path_a() else: path_b()`

El cambio mental que abre la puerta a los agentes: **el LLM como decisor**, no como
generador de texto. Clasifica la pregunta en un dominio y devuelve el motivo. No
responde la pregunta — elige un camino y se detiene.

Se hace con **salida estructurada** (`with_structured_output` sobre un modelo
Pydantic), no parseando texto libre. Vas a usar esto todo el programa.

**Escribes 2 líneas:** los dos campos del modelo Pydantic.

In [ ]:
class Clasificacion(BaseModel):
    """Salida estructurada del router.

    Ojo con las descripciones de los campos: NO son documentación. Viajan al
    modelo como parte del esquema de la función. Son prompt.
    """

    # TODO 1 · Declara el campo `dominio`. Debe aceptar exactamente uno de estos
    #          cuatro valores: mantenimiento, compras, logistica, demanda.
    #          Pista: Literal[...] y Field(description=...)
    dominio: ...

    # TODO 2 · Declara el campo `motivo`: una frase explicando la elección.
    motivo: ...


class MiN2(NivelBase):
    ID = "n2"
    NOMBRE = "Enrutador"
    ESTRELLAS = "★☆☆"
    PATRON = "if llm_decision(): path_a() else: path_b()"

    async def correr(self, ej, pregunta):
        # include_raw=True conserva el mensaje original y con él el conteo de
        # tokens. Sin eso, la fila de costo de N2 quedaría en cero.
        modelo = cliente().with_structured_output(Clasificacion, include_raw=True)
        mensajes = [
            {"role": "system", "content": self.system_prompt()},
            {"role": "user", "content": pregunta},
        ]

        yield ej.ev(
            "llm_request",
            n_llamada=ej.proxima_llamada(),
            mensajes=mensajes_publicables(mensajes),
            tools_declaradas=["Clasificacion"],
        )

        salida = await con_reintentos(lambda: modelo.ainvoke(mensajes))
        uso = salida["raw"].usage_metadata or {}
        ej.contar_tokens(uso.get("input_tokens", 0), uso.get("output_tokens", 0))
        decision = salida["parsed"]

        yield ej.ev(
            "llm_response", n_llamada=1, texto=None, hay_tool_calls=True,
            tokens_in=uso.get("input_tokens", 0),
            tokens_out=uso.get("output_tokens", 0), ms=0,
        )

        yield ej.ev("ruta", dominio=decision.dominio, motivo=decision.motivo)

        ej.respuesta_final = (
            f"Corresponde a {decision.dominio}. {decision.motivo} "
            "(Este nivel solo enruta: no consulta datos.)"
        )
        yield ej.ev("respuesta_final", texto=ej.respuesta_final)


traza = await correr(MiN2(), PREGUNTA)

In [ ]:
# --- Corro y verifico ---
traza = ultima_traza()

assert traza.llamadas_llm == 1, f"N2 hace 1 llamada, hizo {traza.llamadas_llm}"
assert traza.llamadas_tools == 0, "N2 no ejecuta nada, solo decide"
assert traza.ruta in {"mantenimiento", "compras", "logistica", "demanda"}, \
    f"'{traza.ruta}' no está en el enum de cuatro dominios"

print(f"→ Enrutó a: {traza.ruta}")
print("\n✓ N2 correcto")

### Observo

1. Enrutó a `compras`… **¿y ahora quién ejecuta?** Esa pregunta es exactamente la
   entrada a N3.
2. Prueba esta pregunta ambigua: *«El pedido de la avícola no llegó y creo que el
   molino está parado.»* El router **tiene que elegir uno** y con eso pierde la
   mitad del problema. Recuerda esto en N5.

In [ ]:
# --- Reto (opcional, para después de la sesión) ---
#
#   await correr(MiN2(), "El pedido de la avícola no llegó y creo que el molino está parado.")
#
# [NÚCLEO] Agrega un quinto dominio y un caso genuinamente ambiguo. ¿Qué hace el
# router cuando dos dominios empatan? ¿Se nota en el `motivo`?
# Muéstralo a tu mesa.

---

## 3 · N3 · Llamador de herramientas ★★☆

> ⏱ **Minuto 11 de 55.** Si tu N2 no quedó, ejecuta la celda de rescate de abajo
> y sigue. **Este es el nivel más importante de la clase.**

**Patrón:** `run_function(llm_chosen_tool, llm_chosen_args)`

Ahora el modelo puede consultar la base. El ciclo, **escrito a mano**:

1. Llamada al modelo con las herramientas declaradas.
2. Si la respuesta trae `tool_calls`, ejecutas la función.
3. Devuelves el resultado al modelo como mensaje de tool.
4. Segunda llamada. Responde. **Se detiene ahí: máximo una ronda.**

Lo escribes a mano a propósito. Si el framework aparece antes de que entiendas qué
abstrae, es magia. En N4 lo vas a reconocer.

**Escribes 4 líneas.**

In [ ]:
# --- Celda de rescate: N2 de referencia ---
# Si tu N2 no quedó funcionando, ejecuta esta celda y sigue. No te quedes atrás por
# eso — el nivel 3 es lo importante.
from backend.niveles.n2_router import Clasificacion, N2 as MiN2  # noqa: F811
print("N2 de referencia cargado.")

In [ ]:
class MiN3(NivelBase):
    ID = "n3"
    NOMBRE = "Llamador de herramientas"
    ESTRELLAS = "★★☆"
    PATRON = "run_function(llm_chosen_tool, llm_chosen_args)"

    async def correr(self, ej, pregunta):
        openai = cliente_crudo()
        esquemas = esquemas_openai(TOOLS)     # ← la docstring de cada tool va acá
        declaradas = [e["function"]["name"] for e in esquemas]

        mensajes = [
            {"role": "system", "content": self.system_prompt()},
            {"role": "user", "content": pregunta},
        ]

        # ---- Paso 1 · primera llamada, con las tools declaradas -----------
        yield ej.ev("llm_request", n_llamada=ej.proxima_llamada(),
                    mensajes=mensajes_publicables(mensajes),
                    tools_declaradas=declaradas)

        r = await con_reintentos(lambda: openai.chat.completions.create(
            model=config.openai_model, messages=mensajes,
            tools=esquemas, temperature=0,
            # UNA tool por respuesta. Sin esto el modelo pide varias en paralelo,
            # resuelve todo en su única ronda y este nivel deja de tener límite
            # que mostrar — que es justo lo que enseña.
            parallel_tool_calls=False))

        bruto = r.model_dump()
        mensaje = bruto["choices"][0]["message"]
        llamadas = mensaje.get("tool_calls") or []
        uso = bruto.get("usage") or {}
        ej.contar_tokens(uso.get("prompt_tokens", 0), uso.get("completion_tokens", 0))

        yield ej.ev("llm_response", n_llamada=1, texto=mensaje.get("content"),
                    hay_tool_calls=bool(llamadas),
                    tokens_in=uso.get("prompt_tokens", 0),
                    tokens_out=uso.get("completion_tokens", 0), ms=0)

        if not llamadas:
            ej.respuesta_final = mensaje.get("content") or ""
            yield ej.ev("respuesta_final", texto=ej.respuesta_final)
            return

        mensajes.append({"role": "assistant", "content": mensaje.get("content"),
                         "tool_calls": llamadas})

        # ---- Paso 2 · ejecutar lo que pidió el modelo ---------------------
        for llamada in llamadas:
            funcion = llamada["function"]

            # TODO 1 · Saca el nombre de la tool y sus argumentos.
            #          CUIDADO: funcion["arguments"] es un STRING JSON, no un dict.
            nombre = ...
            argumentos = ...

            ej.llamadas_tools += 1
            yield ej.ev("tool_call", id_llamada=llamada["id"], nombre=nombre,
                        argumentos=argumentos,
                        # Esto es lo que se proyecta al lado del ebook, sección 7.2:
                        crudo=json.dumps(llamada, ensure_ascii=False, indent=2))

            # TODO 2 · Ejecuta la función Python que el modelo eligió.
            #          POR_NOMBRE mapea el nombre a la función.
            resultado = ...

            yield ej.ev("tool_result", id_llamada=llamada["id"], nombre=nombre,
                        resultado=resultado, filas=None, ms=0, error=None)

            # TODO 3 · Devuelve el resultado al modelo. Un mensaje de rol "tool"
            #          necesita tool_call_id y content (un string).
            mensajes.append(...)

        # ---- Paso 3 · segunda llamada, con el resultado en contexto -------
        # Sin `tools=`: no queremos otra ronda. Esa limitación ES el nivel.
        yield ej.ev("llm_request", n_llamada=ej.proxima_llamada(),
                    mensajes=mensajes_publicables(mensajes), tools_declaradas=[])

        # TODO 4 · Haz la segunda llamada al modelo.
        r2 = ...

        bruto2 = r2.model_dump()
        mensaje2 = bruto2["choices"][0]["message"]
        uso2 = bruto2.get("usage") or {}
        ej.contar_tokens(uso2.get("prompt_tokens", 0), uso2.get("completion_tokens", 0))

        yield ej.ev("llm_response", n_llamada=2, texto=mensaje2.get("content"),
                    hay_tool_calls=False,
                    tokens_in=uso2.get("prompt_tokens", 0),
                    tokens_out=uso2.get("completion_tokens", 0), ms=0)

        ej.respuesta_final = mensaje2.get("content") or ""
        yield ej.ev("respuesta_final", texto=ej.respuesta_final)


traza = await correr(MiN3(), PREGUNTA)

In [ ]:
# --- Corro y verifico ---
traza = ultima_traza()

assert traza.llamadas_llm == 2, f"N3 hace exactamente 2 llamadas, hizo {traza.llamadas_llm}"
assert traza.llamadas_tools == 1, f"N3 hace 1 ronda de tools, hizo {traza.llamadas_tools}"
assert traza.tool_calls[0].crudo, "Falta el JSON crudo de la tool call"
json.loads(traza.tool_calls[0].crudo)          # debe ser JSON parseable
assert "320" in traza.respuesta_final, \
    f"La respuesta debe traer el valor real de la base. Dijo: {traza.respuesta_final!r}"

print("--- El JSON crudo que devolvió el modelo ---")
print(traza.tool_calls[0].crudo)
print("\n✓ N3 correcto")

### Observo

1. **El JSON crudo.** Compáralo con la sección 7.2 del ebook. `name` y
   `arguments` — y `arguments` es un **string**, no un objeto. Eso es todo lo que
   es *function calling*: el modelo no ejecuta nada, solo dice qué ejecutarías.
2. **La respuesta está correcta y está incompleta.** Dijo cuánto maíz hay, pero no
   si alcanza para la demanda: le faltó consultar una segunda cosa y no le dimos
   un segundo turno. ¿Qué le falta? Una palabra.

In [ ]:
# --- Reto (opcional, para después de la sesión) ---
#
# [NÚCLEO] Haz que el modelo pida DOS tools en la misma respuesta (pregúntale por
# inventario y por el estado de un pedido a la vez). Tu loop las recorre con un
# `for`: ¿las ejecuta en paralelo o en serie? ¿Importa?
# Muéstralo a tu mesa.

---

## 4 · N4 · Agente multipasos (ReAct) ★★★

> ⏱ **Minuto 25 de 55.** Si tu N3 no corre, ejecuta la celda de rescate y sigue a
> N4. En serio: sigue.

**Patrón:** `while llm_should_continue(): execute_next_step()`

La respuesta a «¿qué le falta a N3?» es **iterar**. El ciclo ReAct —Thought →
Action → Observation— repetido hasta que el modelo decida que ya puede responder.

Y aquí aparece el framework. `create_react_agent` hace por debajo exactamente el
loop que acabas de escribir a mano: llamar, detectar `tool_calls`, ejecutar,
reinyectar, repetir. Nada más. No es magia — es tu N3 con un `while`.

> **Un detalle que vale más que la función.** En LangGraph 1.x esta misma
> `create_react_agent` ya está **deprecada**: la API vigente es
> `langchain.agents.create_agent`, y la que estás usando desaparece en la 2.0.
> Funciona, y la usamos porque es la del ebook. Pero para tu proyecto real usa la
> nueva. Que la función que aprendes hoy ya esté marcada para morir es la lección
> más transferible de este bloque: **lo que hay debajo —el loop que escribiste en
> N3— no se deprecó.** Los frameworks se mueven; el patrón se queda.

**Escribes 3 líneas.**

In [ ]:
# --- Celda de rescate: N3 de referencia ---
# Si tu N3 no quedó funcionando, ejecuta esta celda y sigue.
from backend.niveles.n3_tool_caller import N3 as MiN3  # noqa: F811
print("N3 de referencia cargado.")

In [ ]:
# TODO 1 · El system prompt del agente. Tiene que conseguir tres cosas:
#          · que encadene las consultas que necesite;
#          · que NUNCA afirme una cifra que no venga de una herramienta;
#          · que escriba una frase corta de razonamiento antes de cada acción
#            (el "Thought" del patrón ReAct).
PROMPT_N4 = """
...
"""

# TODO 2 · Las herramientas del agente. ¿Todas? ¿Un subconjunto?
MIS_TOOLS = ...

# TODO 3 · Crea el agente ReAct. Tres piezas: el modelo, las tools y el prompt.
#          Pista: create_react_agent(modelo, tools, prompt=...)
mi_agente = ...


class MiN4(NivelBase):
    ID = "n4"
    NOMBRE = "Agente multipasos"
    ESTRELLAS = "★★★"
    PATRON = "while llm_should_continue(): execute_next_step()"

    async def correr(self, ej, pregunta):
        # `stream_react` traduce lo que hace el grafo a eventos del contrato. Es
        # plomería: lo interesante lo escribiste arriba.
        async for evento in stream_react(
            ej,
            tools=MIS_TOOLS,
            system_prompt=PROMPT_N4,
            pregunta=pregunta,
            agente=mi_agente,
            max_iteraciones=8,      # sin tope, un agente se cuelga en clase
        ):
            yield evento

        yield ej.ev("respuesta_final", texto=ej.respuesta_final)


traza = await correr(MiN4(), PREGUNTA)

In [ ]:
# --- Corro y verifico ---
traza = ultima_traza()

assert traza.llamadas_tools >= 2, \
    f"N4 debe encadenar al menos 2 consultas, hizo {traza.llamadas_tools}"
assert "consultar_inventario" in traza.tools_usadas, "Debe consultar el inventario"
assert "consultar_demanda" in traza.tools_usadas, "Debe consultar la demanda"
assert traza.llamadas_llm <= 9, f"Tope desbordado: {traza.llamadas_llm} llamadas"

conclusiones = {"alcanza", "no alcanza", "suficiente", "insuficiente", "faltan", "déficit"}
assert any(c in traza.respuesta_final.lower() for c in conclusiones), \
    "La respuesta debe CONCLUIR, no solo listar dos cifras"

print(f"herramientas usadas: {traza.tools_usadas}")
print("\n✓ N4 correcto")

### Observo

1. **Nadie le dijo que consultara dos tablas.** Decidió solo que necesitaba el
   inventario y también la demanda, y las comparó. Eso es agencia.
2. **La factura.** Mira las llamadas al modelo de N3 y de N4 para la misma
   pregunta. Ese es el costo de iterar — y es el criterio con el que vas a elegir
   el nivel de tu propio proyecto.

In [ ]:
# --- Reto (opcional, para después de la sesión) ---
#
# Pregúntale por una planta que no existe:
#
#   await correr(MiN4(), "¿Cuánto maíz le queda a la planta de Cali?")
#
# La herramienta devuelve vacío. ¿Lo reporta o se lo inventa? Esto cierra el arco
# que abrió N1.
#
# [NÚCLEO] Provoca que una tool falle de verdad (rompe una a propósito) y observa
# la recuperación. Luego haz que falle SIEMPRE: ¿corta el tope de iteraciones?
# Muéstralo a tu mesa.

---

## 5 · N5 · Sistema multiagente (supervisor) ★★★★

> ⏱ **Minuto 39 de 55.** Si tu N4 no quedó, rescate y sigue: N5 se construye
> ENCIMA de N4 y es corto.

**Patrón:** `if llm_trigger(): execute_agent()`

Arquitectura **supervisor con agentes como tools** (ebook 8.1). Y la revelación
del nivel: no es una arquitectura nueva. Es N4 aplicado un nivel más arriba.

1. Instancias el ReAct dos veces, cada uno con su prompt y su subconjunto de tools.
2. Envuelves cada uno en una función con docstring — **un agente expuesto como
   tool es solo una función que por dentro llama a un agente**.
3. Le pasas esas dos funciones como tools a un tercer ReAct.

La pregunta también cambia: tiene que **cruzar los dos dominios**, porque si no,
un supervisor es un N4 caro.

**Escribes 4 líneas.**

In [ ]:
# --- Celda de rescate: N4 de referencia ---
# Si tu N4 no quedó funcionando, ejecuta esta celda y sigue.
from backend.niveles.n4_react import N4 as MiN4  # noqa: F811
print("N4 de referencia cargado.")

In [ ]:
print(f"La pregunta de este nivel:\n  «{PREGUNTA_CRUZADA}»")

# TODO 1 · El sub-agente de abastecimiento. `descripcion` es lo que lee el
#          SUPERVISOR para decidir a quién llamar: es un prompt, no un comentario.
ABASTECIMIENTO = SubAgente(
    nombre="agente_abastecimiento",
    system_prompt="...",
    tools=...,
    descripcion="...",
)

# TODO 2 · El sub-agente de operaciones (mantenimiento y logística).
OPERACIONES = SubAgente(
    nombre="agente_operaciones",
    system_prompt="...",
    tools=...,
    descripcion="...",
)

# TODO 3 · El prompt del supervisor. No consulta datos: coordina y sintetiza.
#          Que formule instrucciones AUTOCONTENIDAS — el sub-agente no ve la
#          conversación con el usuario.
PROMPT_N5 = """
...
"""


class MiN5(NivelBase):
    ID = "n5"
    NOMBRE = "Supervisor multiagente"
    ESTRELLAS = "★★★★"
    PATRON = "if llm_trigger(): execute_agent()"

    async def correr(self, ej, pregunta):
        # El Delegador envuelve cada sub-agente como tool y saca su actividad
        # interna a la superficie (los `sub_evento` que ves anidados en el tablero).
        delegador = Delegador(ej, [ABASTECIMIENTO, OPERACIONES])

        # TODO 4 · Crea el supervisor: un ReAct cuyas tools son los dos agentes.
        #           delegador.tools te da la lista ya envuelta.
        supervisor = ...

        async for evento in delegador.correr(pregunta, PROMPT_N5, agente=supervisor):
            yield evento

        yield ej.ev("respuesta_final", texto=ej.respuesta_final)


traza = await correr(MiN5(), PREGUNTA_CRUZADA)

In [ ]:
# --- Corro y verifico ---
traza = ultima_traza()

assert len(traza.delegaciones) >= 1, "N5 debe delegar en al menos un sub-agente"
assert set(traza.delegaciones) == {"agente_abastecimiento", "agente_operaciones"}, \
    f"La pregunta cruzada necesita los dos dominios. Delegó en: {traza.delegaciones}"

# Que haya delegado no significa que el sub-agente funcionara: un sub-agente puede
# fallar por dentro y el supervisor responder igual, con menos información. Estos
# dos assert son los que lo detectan.
assert not traza.errores, f"Un sub-agente falló por dentro: {traza.errores}"
assert traza.respuesta_final.strip(), "El supervisor debe sintetizar una respuesta"
assert traza.llamadas_tools >= 2, \
    f"Los dos sub-agentes deben consultar datos; hubo {traza.llamadas_tools} tools"

print(f"delegó en      : {traza.delegaciones}")
print(f"llamadas al LLM: {traza.llamadas_llm}   ← compáralo con N4")
print(f"tools           : {traza.llamadas_tools}")
print("\n✓ N5 correcto")

### Observo

1. **Un agente delegó en otro agente**, y el supervisor nunca tocó la base: pidió
   diagnósticos y los sintetizó.
2. **La factura, otra vez.** N5 cuesta del orden de once llamadas al modelo donde
   N4 hacía cinco. El multiagente se justifica cuando hay **separación real de
   dominios y de contextos** — no por sofisticación. Con una sola pregunta de un
   solo dominio, un supervisor es un N4 caro.

In [ ]:
# --- Reto (opcional, para después de la sesión) ---
#
# [NÚCLEO] Agrega un tercer sub-agente (por ejemplo, uno de calidad o de costos).
# Mide las llamadas al modelo antes y después, con la MISMA pregunta. ¿Cuánto
# cuesta un especialista que no se usa?
# Muéstralo a tu mesa.

---

## 6 · Cierre: la tabla y tu propia tool

> ⏱ **Minuto 51 de 55.**

### La tabla

| | N1 | N2 | N3 | N4 | N5 |
|---|---|---|---|---|---|
| Nivel de agencia | ☆☆☆ | ★☆☆ | ★★☆ | ★★★ | ★★★★ |
| Decide algo | no | qué ruta | qué tool | qué tools y cuántas veces | a qué agente |
| Consulta datos | no | no | 1 ronda | N rondas | vía sub-agentes |
| ¿Responde bien la insignia? | inventa | no responde | a medias | sí | sí |
| **Cuándo usarlo** | clasificar o redactar texto | triage, routing | consulta puntual | análisis multi-fuente | dominios separados |

**La última fila es el entregable de la sesión.** Cada Champion debe poder señalar
en qué columna cae su proyecto y por qué. La interfaz tipo aeropuerto de Logística,
por ejemplo, es un **N3 bien hecho**: una pregunta, una herramienta, una respuesta.
No necesita un agente que itere, y ponerle uno sería pagar diez veces por lo mismo.

Corre la celda siguiente para ver los números de **tus** ejecuciones.

In [ ]:
from backend.llm import estado_lab

print(f"{'nivel':<6} {'llamadas':>9} {'tokens':>10} {'costo':>12}")
for nivel_id, m in sorted(estado_lab.por_nivel.items()):
    costo = f"${m['costo_usd']:.4f}" if config.costo_configurado else "no configurado"
    print(f"{nivel_id:<6} {m['llamadas']:>9} "
          f"{m['tokens_in'] + m['tokens_out']:>10} {costo:>12}")

base = estado_lab.por_nivel.get("n1", {})
unidad = base.get("tokens_in", 0) + base.get("tokens_out", 0)
if unidad:
    print("\ncosto relativo a N1, en tokens:")
    for nivel_id, m in sorted(estado_lab.por_nivel.items()):
        print(f"  {nivel_id}: {(m['tokens_in'] + m['tokens_out']) / unidad:.1f}×")

### Tu propia tool — el entregable para la Sesión 2

Este es el ejercicio más valioso del notebook y **no se hace en clase**: se hace
esta semana y se trae al recap.

> **Escribe la tool que tu proyecto necesita** y conéctala al agente de N4.
>
> 1. Elige una pregunta real de tu proyecto que hoy nadie pueda responder rápido.
> 2. Escribe la función Python que la responde consultando `bios_ops.db`.
> 3. Escribe su docstring **pensando en el modelo**, no en un desarrollador.
> 4. Agrégala a la lista de tools de tu N4 y pregúntale al agente.
> 5. **¿La usó?** Si no la usó, la docstring es la sospechosa. Reescríbela.

El paso 5 es el que enseña. Que un agente ignore una tool porque su descripción es
mala es la lección más transferible de esta sesión, y hay que provocarla.

In [ ]:
# --- Esqueleto. Descomenta el de tu dominio y complétalo. -------------------

# MANTENIMIENTO — predicción de fallas
# def riesgo_de_falla(planta: str, dias: int = 90) -> dict:
#     """Estima qué equipos de una planta tienen mayor riesgo de falla.
#     Cruza correctivos recientes con la tendencia de vibración de cada equipo.
#     Devuelve los equipos ordenados por riesgo, con la evidencia de cada uno.
#     Args:
#         planta: Nombre, municipio o código de la planta.
#         dias: Ventana de historial a considerar.
#     """
#     filas = consulta_directa("SELECT ...", (...,))
#     return {"planta": planta, "equipos": [...]}

# COMPRAS — asignación de volúmenes a plantas
# def sugerir_compra(materia_prima: str, dias_cobertura: int = 30) -> dict:
#     """Calcula cuánto comprar de una materia prima y para qué planta..."""

# LOGÍSTICA — interfaz tipo aeropuerto
# def tablero_cliente(cliente: str) -> dict:
#     """Estado de todos los pedidos de un cliente, como un tablero de vuelos..."""

# PRODUCCIÓN Y TD — planeación de la demanda
# def brecha_demanda_produccion(planta: str, dias: int = 30) -> dict:
#     """Días en que la producción no cubrió la demanda, y de cuánto fue la brecha..."""


# Cuando la tengas, agrégala y vuelve a preguntar:
#
#   MIS_TOOLS = TOOLS + [mi_tool]
#   mi_agente = create_react_agent(cliente(), MIS_TOOLS, prompt=PROMPT_N4)
#   await correr(MiN4(), "…tu pregunta real…")
#
#   ultima_traza().tools_usadas      # ¿aparece la tuya?

---

### Qué sigue

| Sesión | Y este notebook queda como |
|---|---|
| 2 · RAG y memoria | los agentes que ya construiste, ahora con contexto recuperado |
| 3 · MCP y Skills | las tools de hoy, expuestas por protocolo en vez de importadas |
| 5 · Testing y evals | los `assert` de este notebook son tu primer *eval* |
| 6 · Spec Driven Development | las specs de este laboratorio son el caso de estudio |
| 7 · Producción | la tool como superficie de ataque, y el costo como restricción real |

<div align="center">
  <strong>Qypher · Formación en Inteligencia Artificial</strong>
</div>